# GIU Agent

Our own build on top of Day 2/3: one agent that will end up wired to two
tool packages the doctor hands out in the room:

- **portal** — `get_transcript`, `available_grades`, `get_grade_by_name`
- **cms** — `list_course`, `find_course`, `get_content`

Idea: a course advisor agent that cross-references what a student has
already taken (portal) with what's available (cms) to recommend what to
take next, and explain why.

This notebook starts bare, same as `day2-build-an-agent.ipynb`. Tools get
added the moment the packages arrive: nothing else about the call changes.

## 0 · Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

MODEL = "anthropic:claude-haiku-4-5"   # swap provider here if needed
llm = init_chat_model(MODEL)

llm.invoke("Say 'ready' and nothing else.").content

## 1 · The bare agent

No tools yet, just the persona. This is what we grow once the portal/cms
packages land.

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

SYSTEM_PROMPT = """You are the GIU course advisor assistant.
- Ground every claim in a tool result. Never invent a grade, course, or transcript entry.
- If you don't have enough information yet, say what tool call you'd need instead of guessing.
- Be concise and cite the course code when recommending something."""

agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT,
)

result = agent.invoke({"messages": [
    HumanMessage("What courses would you recommend for me next semester?")
]})
print(result["messages"][-1].content)

Expected: it should refuse to guess (no tools yet, so no transcript, no
catalog). Read the trace below to confirm it isn't inventing anything.

In [ ]:
for m in result["messages"]:
    body = m.content if m.content else getattr(m, "tool_calls", "")
    print(m.type.upper().ljust(6), "\u2192", str(body)[:120])

## 2 · Tools (waiting on the doctor's packages)

TODO once `portal` and `cms` are provided:

```python
# from portal import get_transcript, available_grades, get_grade_by_name
# from cms import list_course, find_course, get_content
#
# agent = create_agent(
#     model=llm,
#     system_prompt=SYSTEM_PROMPT,
#     tools=[get_transcript, available_grades, get_grade_by_name,
#            list_course, find_course, get_content],
# )
```